# Cell Tower Coverage

This notebook aims to emulate the Gurobi Cell Tower Coverage (link) using OptVerse.
This example demonstrates solving a cell tower placement problem to provide signal coverage to the largest number of people possible.

## Prerequisites

The repo containing this, and other examples is available [through CodeHub](link)

## Problem Description

The following section is taken verbatim from [Gurobi Cell Tower Coverage example](link/to/cell_tower.ipynb).

A telecom company needs to build a set of cell towers to provide signal coverage for the inhabitants of a given city. A number of potential locations where the towers could be built have been identified. The towers have a fixed range, and -due to budget constraints- only a limited number of them can be built. Given these restrictions, the company wishes to provide coverage to the largest percentage of the population possible. To simplify the problem, the company has split the area it wishes to cover into a set of regions, each of which has a known population. The goal is then to choose which of the potential locations the company should build cell towers on -in order to provide coverage to as many people as possible.

The Cell Tower Coverage Problem is an instance of the Maximal Covering Location Problem. It is also related to the Set Cover Problem.

## Solution Approach

The following section is taken verbatim from [Gurobi Cell Tower Coverage example](link/to/cell_tower.ipynb).

Mathematical programming is a declarative approach where the modeler formulates a mathematical optimization model that captures the key aspects of a complex decision problem. The OptVerse Optimizer solves such models using state-of-the-art mathematics and computer science.

A mathematical optimization model has five components, namely:

* Sets and indices.
* Parameters.
* Decision variables.
* Objective function(s).
* Constraints.

We now present a mixed-integer programming (MIP) formulation for the Cell Tower Coverage Problem.

## Python Implementation

We will now illustrate how to solve the problem described through OptVerse.

### Environment Configuration

To use the OptVerse Python module the user must import the package.

In [ ]:
from optvpy import *

### Data Preparation

We will first prepare the data required to construct the model.
For pedagogical purposes, we will just construct this data explicitly.
This example considers a bipartite graph for 6 towers and 9 regions.

In [ ]:
# Parameters
budget = 20
regions = [0, 1, 2, 3, 4, 5, 6, 7, 8]
population = [523, 690, 420, 1010, 1200, 850, 400, 1008, 950]

sites = [0, 1, 2, 3, 4, 5]
coverage = {
    0: [0, 1, 5],
    1: [0, 7, 8],
    2: [2, 3, 4, 6],
    3: [2, 5, 6],
    4: [0, 2, 6, 7, 8],
    5: [3, 4, 8]
}
cost = [4.2, 6.1, 5.2, 5.5, 4.8, 9.2]

num_sites = len(sites)
num_regions = len(regions)

# Create coverage matrix for constraints
coverage_matrix = []
for r in regions:
    covering_sites = []
    for s in sites:
        if r in coverage[s]:
            covering_sites.append(s)
    coverage_matrix.append(covering_sites)

### Model Creation

To create a model, we must first instantiate an environment, and pass it to the constructor for our model object.

In [ ]:
env = OPTVEnv()
model = OPTVModel(env)

### Add Variables

Next, we add our 'build' and 'is_covered' decision variables to our model.
We set the linear coefficients to the objective through the variables addition interface.

In [ ]:
# Add the build variables
build_lb = num_sites * [0]
build_ub = num_sites * [1]
build_obj = num_sites * [0]  # Not directly in objective for maximization
build_type = num_sites * [OPTV_BINARY]
build_name = [f"build[{i}]" for i in range(num_sites)]

build = model.AddVars(build_lb, build_ub, build_obj, build_type, build_name)

# Add the is_covered variables
covered_lb = num_regions * [0]
covered_ub = num_regions * [1]
covered_obj = population  # Maximize population covered
covered_type = num_regions * [OPTV_BINARY]
covered_name = [f"covered[{i}]" for i in range(num_regions)]

is_covered = model.AddVars(covered_lb, covered_ub, covered_obj, covered_type, covered_name)

### Add Constraints

Now we add the constraints to our model.

In [ ]:
# Coverage constraints: sum of towers covering region r >= is_covered[r]
coverage_expr = [sum([build[s] for s in coverage_matrix[r]]) - is_covered[r] for r in range(num_regions)]
coverage_lb = num_regions * [0]
coverage_ub = num_regions * [OPTV_INF]
coverage_name = [f"coverage[{i}]" for i in range(num_regions)]

model.AddConstrs(coverage_expr, coverage_lb, coverage_ub, coverage_name)

# Budget constraint: sum of costs <= budget
budget_expr = [sum([cost[s] * build[s] for s in range(num_sites)])]
budget_lb = [-OPTV_INF]
budget_ub = [budget]
budget_name = ["budget"]

model.AddConstrs(budget_expr, budget_lb, budget_ub, budget_name)

### Set Objective

Set the objective to maximize population coverage.

In [ ]:
model.SetObjectiveSense(OPTV_MAXIMIZE)

### Optimize the Model

Finally, we can run the optimization on the problem.

In [ ]:
model.Optimize()

## Analyze the Solution

The result of the optimization model shows the maximum population that can be covered with the $20,000,000 budget.
Let's see the solution that achieves that optimal result.

This plan determines at which site locations to build a cell tower.

In [ ]:
for tower in build:
    if (abs(tower.Get(OPTVDblAttr.X)) > 1e-6):
        print(f"\n Build a cell tower at location Tower {tower.Index()}.")

### Coverage Plan

This plan determines which regions are covered by the towers built.

In [ ]:
for i in range(num_regions):
    if (abs(is_covered[i].Get(OPTVDblAttr.X)) > 1e-6):
        print(f"\n Region {i} with population {population[i]} is covered.")

### Performance Metrics

Calculate coverage and budget utilization metrics.

In [ ]:
# Calculate total population covered
total_population = sum(population)
total_covered = 0

for i in range(num_regions):
    if (abs(is_covered[i].Get(OPTVDblAttr.X)) > 1e-6):
        total_covered += population[i]

coverage_percentage = round(100 * total_covered / total_population, 2)
print(f"\n The population coverage associated to the cell towers build plan is: {coverage_percentage} %")

# Calculate budget consumption
total_cost = 0
for tower in build:
    if (abs(tower.Get(OPTVDblAttr.X)) > 1e-6):
        total_cost += cost[tower.Index()]

budget_consumption = round(100 * total_cost / budget, 2)
print(f"\n The percentage of budget consumed associated to the cell towers build plan is: {budget_consumption} %")

## Conclusion

In this example, we addressed a cell tower coverage problem where we want to build cell towers to provide signal coverage to the largest number of people possible while satisfying a budget constraint.
We learned how to formulate the problem as a MIP model.
Also, we learned how to implement the MIP model formulation and solve it using the OptVerse Python API.